# CHỦ ĐỀ: CÂY TÌM KIẾM CÂN BẰNG 2-3 (2-3 SEARCH TREE)

---

## 1. Bài toán 

### 1.1. Thuật toán giải quyết bài toán gì?
* **Bài toán Bảng ký hiệu có thứ tự (Ordered Symbol Table / Dictionary Problem):** Quản lý một tập hợp dữ liệu động hỗ trợ hiệu quả các thao tác: **Tìm kiếm (`search`)**, **Chèn (`insert`)**, **Xóa (`delete`)** và truy vấn theo thứ tự.
* **Hạn chế của Cây nhị phân tìm kiếm (BST) truyền thống:** 
  * BST thông thường có độ phức tạp trung bình là $\mathcal{O}(\log N)$, nhưng phụ thuộc hoàn toàn vào thứ tự dữ liệu đầu vào.
  * Trong **trường hợp xấu nhất** (dữ liệu đầu vào đã sắp xếp tăng/giảm dần), BST bị suy biến thành danh sách liên kết đơn (chiều cao $h = N$), khiến mọi thao tác chậm xuống mức $\mathcal{O}(N)$.
* **Mục tiêu của Cây 2-3:** Đảm bảo cây **cân bằng tuyệt đối (perfectly balanced)** mọi lúc, giữ cho chiều cao cây luôn đạt $\mathcal{O}(\log N)$ trong cả trường hợp xấu nhất.

### 1.2. Đầu vào và Đầu ra
* **Đầu vào (Input):** 
  * Một dãy các khóa $K = \{k_1, k_2, \dots, k_n\}$ có thể so sánh được (Comparable keys) cùng các yêu cầu thao tác (`insert(k)`, `search(k)`).
* **Đầu ra (Output):**
  * Cấu trúc **Cây 2-3 hợp lệ** (mọi nút lá nằm trên cùng một mức, các khóa phân bố đúng thứ tự từ điển).
  * Kết quả truy vấn: trả về `True` (hoặc giá trị gắn với khóa) nếu tìm thấy, `False` (hoặc `None`) nếu không tồn tại.

---

## 2. Ý tưởng cốt lõi 

### 2.1. Cấu trúc nút linh hoạt (Cho phép nút "béo" lên)
Thay vì ép mỗi nút chỉ được chứa đúng 1 khóa như BST, Cây 2-3 cho phép tồn tại **2 loại nút**:

| Loại nút | Số khóa (Keys) | Số nhánh con (Links) | Quy tắc phân hoạch khoảng giá trị |
| :--- | :---: | :---: | :--- |
| **Nút 2 (2-node)** | $1$ khóa ($K_1$) | $2$ con (`left`, `right`) | $\text{left} < K_1 < \text{right}$ |
| **Nút 3 (3-node)** | $2$ khóa ($K_1 < K_2$) | $3$ con (`left`, `mid`, `right`) | $\text{left} < K_1 < \text{mid} < K_2 < \text{right}$ |

* **Bất biến cân bằng hoàn hảo (Perfect Balance Invariant):** Mọi đường đi từ nút gốc (root) đến tất cả các nút lá (`null` links) đều có **cùng độ dài**.
* Chiều cao $H$ của cây 2-3 chứa $N$ khóa luôn thỏa mãn:
  $$\log_3(N + 1) - 1 \le H \le \log_2(N + 1) - 1$$

### 2.2. Chiến lược chính: Phát triển từ dưới lên (Bottom-up Growth & Local Split)
* **Tại sao BST bị lệch?** Vì BST phát triển **từ trên xuống (Top-down)**: khóa mới luôn bị gắn thêm làm lá mới ở đáy cây, làm tăng chiều cao cục bộ.
* **Chiến lược của Cây 2-3:** Không bao giờ tạo nút lá mới ở dưới đáy!
  1. Tìm xuống nút lá phù hợp và **nhét thẳng khóa mới vào nút lá đó**.
  2. Nếu nút lá đang là **2-node** (1 khóa) $\rightarrow$ trở thành **3-node** (2 khóa). Xong!
  3. Nếu nút lá đang là **3-node** (2 khóa) $\rightarrow$ tạm thời tràn thành **4-node** (chứa 3 khóa: $S < M < L$).
  4. **Phép tách cục bộ (Local Split):** Tách 4-node tạm thời thành hai 2-node chứa $S$ và $L$, đồng thời **đẩy khóa ở giữa ($M$) lên nút cha**.
  5. Quá trình đẩy khóa giữa có thể lan truyền ngược lên tới gốc. Nếu **nút gốc** bị tràn thành 4-node và bị tách, chiều cao của toàn bộ cây mới tăng thêm $1$ bậc cùng lúc!
  

## 3. Minh họa bằng tay (Trace từng bước)

**Bài toán ví dụ:** Chèn lần lượt dãy gồm 7 khóa tăng dần: **`10, 20, 30, 40, 50, 60, 70`** vào một Cây 2-3 ban đầu rỗng, sau đó thực hiện tìm kiếm khóa **`30`** và **`25`**.

> **Tại sao chọn dãy tăng dần?** 
> Nếu dùng Cây nhị phân tìm kiếm (BST) thông thường, dãy tăng dần `10 -> 70` sẽ làm cây bị suy biến hoàn toàn thành một danh sách liên kết lệch phải có chiều cao $H = 6$. Hãy quan sát cách **Cây 2-3** xử lý tình huống xấu nhất này để giữ chiều cao chỉ ở mức $H = 2$.

---

### 3.1. Trace từng bước quá trình Chèn (Insertion)

#### 🔹 Bước 1: Chèn `10` và Bước 2: Chèn `20`
* **Chèn `10`:** Cây đang rỗng $\rightarrow$ khởi tạo nút gốc là một **2-node** chứa khóa `[ 10 ]`.
* **Chèn `20`:** Tìm đến nút lá `[ 10 ]`. Vì nút này mới có 1 khóa nên ta chèn trực tiếp `20` vào cùng nút $\rightarrow$ nâng cấp thành **3-node** `[ 10 | 20 ]`.

| Trạng thái sau Bước 1 (Chèn `10`) | Trạng thái sau Bước 2 (Chèn `20`) |
| :---: | :---: |
| **`[ 10 ]`** *(2-node)* | **`[ 10 | 20 ]`** *(3-node)* |

---

#### 🔹 Bước 3: Chèn `30` *(Xuất hiện phép tách 4-node đầu tiên)*
1. **Chèn vào lá:** Đưa `30` vào nút lá hiện tại $\rightarrow$ nút bị tràn tạm thời thành **4-node** `[ 10 | 20 | 30 ]`.
2. **Tách nút (Split):** 
   * Đẩy khóa ở giữa là **`20`** lên trên để tạo thành nút gốc mới.
   * Tách hai khóa còn lại thành hai nút con **2-node**: con trái `[ 10 ]` và con phải `[ 30 ]`.
   * *Chiều cao của cây tăng từ $0$ lên $1$.*

| 1. Tràn tạm thời (4-node) | $\Longrightarrow$ | 2. Sau khi tách (Split) |
| :---: | :---: | :---: |
| **`[ 10 | 20 | 30 ]`** | **Đẩy `20` lên gốc** | **Gốc:** **`[ 20 ]`**<br>↙️ &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; ↘️<br>**`[ 10 ]`** &nbsp;&nbsp;&nbsp;&nbsp;&nbsp; **`[ 30 ]`** |

---

#### 🔹 Bước 4: Chèn `40` *(Hấp thụ vào nút lá 2-node)*
1. **Tìm đường đi:** Bắt đầu từ gốc `[ 20 ]`, do $40 > 20$ nên đi xuống nhánh con bên phải là `[ 30 ]`.
2. **Chèn vào lá:** Nút lá `[ 30 ]` đang là 2-node (còn chỗ trống) $\rightarrow$ nhận thêm `40` để trở thành **3-node** `[ 30 | 40 ]`. Cây giữ nguyên chiều cao.

> **Cấu trúc cây sau Bước 4:**
> * **Mức 0 (Gốc):** **`[ 20 ]`**
>   * **Nhánh Trái ($< 20$):** `[ 10 ]` *(2-node)*
>   * **Nhánh Phải ($> 20$):** **`[ 30 | 40 ]`** *(3-node)*

---

#### 🔹 Bước 5: Chèn `50` *(Tách nút lá, đẩy khóa giữa lên nút cha 2-node)*
1. **Tìm đường đi:** Từ gốc `[ 20 ]`, vì $50 > 20$ nên đi xuống nút lá phải `[ 30 | 40 ]`.
2. **Tràn tại lá:** Chèn `50` làm nút lá phải tràn thành **4-node tạm thời** `[ 30 | 40 | 50 ]`.
3. **Tách lá & Đẩy lên cha:**
   * Tách `[ 30 | 40 | 50 ]` thành hai 2-node là `[ 30 ]` và `[ 50 ]`.
   * Đẩy khóa giữa **`40`** lên nút cha `[ 20 ]` $\rightarrow$ nút cha từ 2-node biến thành **3-node** **`[ 20 | 40 ]`** quản lý 3 nhánh con (Trái, Giữa, Phải)!

| 1. Tràn tại nhánh phải | $\Longrightarrow$ | 2. Đẩy `40` lên cha `[ 20 ]` |
| :---: | :---: | :---: |
| **Gốc:** `[ 20 ]`<br>↙️ &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; ↘️<br>`[ 10 ]` &nbsp;&nbsp; **`[ 30 | 40 | 50 ]`** | **Tách lá phải** | **Gốc (3-node):** **`[ 20 | 40 ]`**<br>↙️ &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; ⬇️ &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; ↘️<br>**`[ 10 ]`** &nbsp;&nbsp; **`[ 30 ]`** &nbsp;&nbsp; **`[ 50 ]`** |

---

#### 🔹 Bước 6: Chèn `60` *(Hấp thụ vào nhánh phải cùng)*
1. **Tìm đường đi:** So sánh với gốc `[ 20 | 40 ]`, vì $60 > 40$ nên rẽ xuống nhánh con phải cùng là `[ 50 ]`.
2. **Chèn vào lá:** Nút `[ 50 ]` đang là 2-node $\rightarrow$ hấp thụ `60` và trở thành **3-node** **`[ 50 | 60 ]`**.

> **Cấu trúc cây sau Bước 6:**
> * **Mức 0 (Gốc):** **`[ 20 | 40 ]`** *(3-node)*
>   * **Con Trái ($< 20$):** `[ 10 ]`
>   * **Con Giữa ($20..40$):** `[ 30 ]`
>   * **Con Phải ($> 40$):** **`[ 50 | 60 ]`** *(3-node

In [1]:
class Node23:
    
    def __init__(self, key=None):
        self.keys = [] if key is None else [key]
        self.parent = None
        self.children = []

    def is_leaf(self):
        return len(self.children) == 0

    def is_full(self):
        return len(self.keys) == 3

In [2]:
from Node import Node23
from Insertion import Insert
from InOrderTranversal import InOrderTranversal
from Deletion import Delete
from Search import Search


class TwoThreeTree(Search, Insert, Delete, InOrderTranversal):
    def __init__(self):
        self.root = None

In [3]:
from Node import Node23

class Search:
    
    def search(self, key):
            return self._search(self.root, key)
    
    def _search(self, node, key):
            if node is None:
                return False
    
            if key in node.keys:
                return True
    
            if node.is_leaf():
                return False
    
            if key < node.keys[0]:
                return self._search(node.children[0], key)
    
            if len(node.keys) == 1 or key < node.keys[1]:
                return self._search(node.children[1], key)
    
            return self._search(node.children[2], key)

In [4]:
from Node import Node23

class Insert:
    
    def insert(self, key):
        if self.root is None:
            self.root = Node23(key)
            return

        # Không chèn khóa trùng
        if self.search(key):
            return

        leaf = self._find_leaf(self.root, key)
        self._insert_into_leaf(leaf, key)

    def _find_leaf(self, node, key):
        if node.is_leaf():
            return node

        if key < node.keys[0]:
            return self._find_leaf(node.children[0], key)

        if len(node.keys) == 1 or key < node.keys[1]:
            return self._find_leaf(node.children[1], key)

        return self._find_leaf(node.children[2], key)

    def _insert_into_leaf(self, leaf, key):
        leaf.keys.append(key)
        leaf.keys.sort()

        if leaf.is_full():
            self._split(leaf)

    def _split(self, node):
        # node có 3 khóa: [a, b, c]
        left_key = node.keys[0]
        mid_key = node.keys[1]
        right_key = node.keys[2]

        left_node = Node23(left_key)
        right_node = Node23(right_key)

        # Nếu là nút trong, chia lại 4 cây con
        if not node.is_leaf():
            left_node.children = node.children[:2]
            right_node.children = node.children[2:]

            for child in left_node.children:
                child.parent = left_node

            for child in right_node.children:
                child.parent = right_node

        parent = node.parent

        # Tách nút gốc
        if parent is None:
            new_root = Node23(mid_key)
            new_root.children = [left_node, right_node]

            left_node.parent = new_root
            right_node.parent = new_root

            self.root = new_root
            return

        # Thay node bằng hai nút mới
        idx = parent.children.index(node)

        parent.children[idx] = left_node
        parent.children.insert(idx + 1, right_node)

        left_node.parent = parent
        right_node.parent = parent

        # Đưa khóa giữa lên cha
        parent.keys.append(mid_key)
        parent.keys.sort()

        # Sắp xếp các cây con theo thứ tự khóa
        parent.children.sort(
            key=lambda child: min(child.keys)
        )

        if parent.is_full():
            self._split(parent)

In [ ]:
from Node import Node23

class Delete: 
    
    def delete(self, key):
        node = self._find_node(self.root, key)

        if node is None:
            return False

        # Nếu là nút trong, thay bằng successor
        if not node.is_leaf():
            idx = node.keys.index(key)
            successor_leaf, successor_key = self._get_successor(
                node, idx
            )

            node.keys[idx] = successor_key
            node = successor_leaf
            key = successor_key

        node.keys.remove(key)

        if len(node.keys) == 0:
            self._fix_underflow(node)

        return True

    def _find_node(self, node, key):
        if node is None:
            return None

        if key in node.keys:
            return node

        if node.is_leaf():
            return None

        if key < node.keys[0]:
            return self._find_node(node.children[0], key)

        if len(node.keys) == 1 or key < node.keys[1]:
            return self._find_node(node.children[1], key)

        return self._find_node(node.children[2], key)

    def _get_successor(self, node, key_idx):
        curr = node.children[key_idx + 1]

        while not curr.is_leaf():
            curr = curr.children[0]

        return curr, curr.keys[0]

    def _fix_underflow(self, node):
        # Trường hợp nút gốc bị rỗng
        if node == self.root:
            if len(node.keys) == 0:
                if node.children:
                    self.root = node.children[0]
                    self.root.parent = None
                else:
                    self.root = None
            return

        parent = node.parent
        idx = parent.children.index(node)

        left_sib = (
            parent.children[idx - 1] if idx > 0 else None
        )
        right_sib = (
            parent.children[idx + 1]
            if idx + 1 < len(parent.children)
            else None
        )

        # 1. Mượn khóa từ anh em bên trái
        if left_sib is not None and len(left_sib.keys) == 2:
            node.keys.insert(0, parent.keys[idx - 1])
            parent.keys[idx - 1] = left_sib.keys.pop()

            if left_sib.children:
                child = left_sib.children.pop()
                node.children.insert(0, child)
                child.parent = node

            return

        # 2. Mượn khóa từ anh em bên phải
        if right_sib is not None and len(right_sib.keys) == 2:
            node.keys.append(parent.keys[idx])
            parent.keys[idx] = right_sib.keys.pop(0)

            if right_sib.children:
                child = right_sib.children.pop(0)
                node.children.append(child)
                child.parent = node

            return

        # 3. Không mượn được: gộp với anh em bên trái
        if left_sib is not None:
            parent_key = parent.keys.pop(idx - 1)
            parent.children.pop(idx)

            left_sib.keys.append(parent_key)
            left_sib.keys.extend(node.keys)

            for child in node.children:
                child.parent = left_sib

            left_sib.children.extend(node.children)

        # 4. Không có anh em trái: gộp với anh em bên phải
        elif right_sib is not None:
            parent_key = parent.keys.pop(idx)
            parent.children.pop(idx + 1)

            node.keys.append(parent_key)
            node.keys.extend(right_sib.keys)

            for child in right_sib.children:
                child.parent = node

            node.children.extend(right_sib.children)

        # Nếu cha bị rỗng, xử lý tiếp từ cha
        if len(parent.keys) == 0:
            self._fix_underflow(parent)


In [ ]:
from Node import Node23

class InOrderTranversal:
    
    def in_order(self):
        result = []
        self._traverse(self.root, result)
        return result

    def _traverse(self, node, result):
        if node is None:
            return

        if node.is_leaf():
            result.extend(node.keys)
            return

        for i, key in enumerate(node.keys):
            self._traverse(node.children[i], result)
            result.append(key)

        self._traverse(node.children[len(node.keys)], result)


### 4.2. Giải thích các phần quan trọng trong Code Python 

1. **Cách biểu diễn nút (`class Node`):**
   * Thay vì tách riêng `class TwoNode` và `class ThreeNode` (gây phức tạp khi phải ép kiểu qua lại), ta dùng danh sách động `self.keys` và `self.children`.
   * Điều này cho phép một nút **tạm thời giữ 3 khóa và 4 con** (`is_overflow() == True`) ngay trước khi bị tách, khớp hoàn toàn với khái niệm **4-node tạm thời** trong lý thuyết.
2. **Cơ chế đẩy khóa ngược lên trên trong `_insert()`:**
   * Sau lời gọi đệ quy `self._insert(node.children[child_idx], key)`, nút cha kiểm tra ngay xem con của mình có bị tràn không.
   * Dòng lệnh `node.keys.insert(child_idx, mid_key)` và `node.children[child_idx:child_idx + 1] = [left_c, right_c]` thực hiện phép biến đổi локаl: đẩy khóa giữa lên cha và nối 2 nhánh mới vào đúng vị trí cũ chỉ trong $\mathcal{O}(1)$ phép toán.
3. **Độ phức tạp thời gian & không gian:**
   * **Tìm kiếm (`search`) & Chèn (`insert`):** Đều tỉ lệ thuận với chiều cao cây $H \le \log_2 N$, do đó đảm bảo độ phức tạp **$\mathcal{O}(\log N)$ trong trường hợp xấu nhất**.
   * **Ý nghĩa thực tiễn:** Cây 2-3 chính là tiền đề trực tiếp để phát triển **Cây Đỏ-Đen (Red-Black Tree)** (dùng trong `std::map` của C++, `TreeMap` của Java) và **B-Tree** (dùng trong cơ sở dữ liệu SQL và hệ thống file).